# 🌧️ Analisis Curah Hujan Multi-Skala: Satelit Kebumen vs AWS IoT Jerukagung
### 📍 Evaluasi Presipitasi Resolusi Per Jam (*Hourly*) & Resolusi Per Hari (*Daily*) Menggunakan `Data_Curah_Hujan_Kebumen.csv`

---
### 📌 Ringkasan Eksekutif
Notebook ini membandingkan data presipitasi satelit & reanalisis pada dua domain waktu:
1. **Data Harian**: Menggunakan dataset `Data_Curah_Hujan_Kebumen.csv` (8 produk: `CHIRPS_RNL`, `CHIRPS_SAT`, `CHIRPS_FNL`, `GSMaP`, `IMERG`, `PERSIANN`, `ERA5`, `ERA5_LAND`) vs agregasi harian AWS IoT Jerukagung.
2. **Data Per Jam**: Menggunakan dataset resolusi 1-jam sinkron (`id-05_clear_data_hourly.csv`, GSMaP, IMERG, ERA5 Hourly).


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

print("Library dan environment berhasil dimuat.")


Library dan environment berhasil dimuat.


## 📂 1. Pemuatan Dataset Harian (8 Satelit Kebumen) & Jam-jaman AWS IoT

In [4]:
data_dir = r'../Data_Satelit'
if not os.path.exists(data_dir):
    data_dir = r'd:/Github/Projek_Rainfall/Google_Earth_Engine/Data_Satelit'

# 1. Dataset Harian Kebumen
df_kebumen = pd.read_csv(os.path.join(data_dir, 'Data_Curah_Hujan_Kebumen.csv'))
df_kebumen['Date'] = pd.to_datetime(df_kebumen['datetime_utc'] if 'datetime_utc' in df_kebumen.columns else df_kebumen['Date'])
sat_cols = ['CHIRPS_RNL', 'CHIRPS_SAT', 'CHIRPS_FNL', 'GSMaP', 'IMERG', 'PERSIANN', 'ERA5', 'ERA5_LAND']
df_daily_sat = df_kebumen.set_index('Date')[sat_cols]

# 2. Dataset Jam-jaman AWS IoT
df_aws = pd.read_csv(os.path.join(data_dir, 'id-05_clear_data_hourly.csv'))
df_aws['Date'] = pd.to_datetime(df_aws['datetime_utc'])
df_aws_daily = pd.DataFrame()
df_aws_daily['rain_aws'] = df_aws.set_index('Date')['rainrate'].resample('D').apply(lambda s: s.sum(min_count=20))

# Gabung Data Harian Master
df_daily_master = df_daily_sat.join(df_aws_daily, how='inner').dropna(subset=['rain_aws'])

print(f"Total Hari Valid Overlap (8 Satelit Kebumen vs AWS IoT Harian): {len(df_daily_master):,} hari")
display(df_daily_master.head())


Total Hari Valid Overlap (8 Satelit Kebumen vs AWS IoT Harian): 563 hari


,CHIRPS_RNL,CHIRPS_SAT,CHIRPS_FNL,GSMaP,IMERG,PERSIANN,ERA5,ERA5_LAND,rain_aws
Date,,,,,,,,,
2025-01-01,26.584682,15.097500,24.008106,9.472854,26.610000,22.183128,6.038189,6.341606,42.483026
2025-01-02,3.851536,0.032677,0.000000,1.674022,0.095000,1.359381,0.181198,0.629991,0.869627
2025-01-03,2.131236,0.695486,0.000000,0.000000,1.610000,0.000000,1.448870,0.457051,0.000000
2025-01-04,21.199074,41.536079,24.008106,14.132218,46.944999,10.235415,8.358717,23.115918,58.232112
2025-01-05,12.832295,9.237086,12.004053,16.577654,16.445000,13.325383,11.823654,12.690067,33.236679


## 📊 2. Ringkasan Metrik Evaluasi: Per Jam vs Per Hari

In [6]:
df_eval_d = pd.read_csv(r'Hasil_Analisis/ringkasan_evaluasi_harian_8satelit_vs_aws.csv')
print("=== EVALUASI 8 PRODUK SATELIT HARIAN VS AWS IOT ===")
display(df_eval_d)

df_multi = pd.read_csv(r'Hasil_Analisis/ringkasan_multi_skala_jam_vs_hari.csv')
print("=== PERBANDINGAN MULTI-SKALA (JAM VS HARI) ===")
display(df_multi)


=== EVALUASI 8 PRODUK SATELIT HARIAN VS AWS IOT ===
=== PERBANDINGAN MULTI-SKALA (JAM VS HARI) ===


,N,Pearson_r,Spearman_rho,RMSE,MAE,MBE,PBIAS,NSE,KGE,IOA,Produk,Skala,Satuan
0,563,0.625043,0.641259,11.052420,6.461952,-0.986568,-11.664178,0.346298,0.569257,0.773789,CHIRPS_SAT,Per Hari (Daily),mm/hari
1,563,0.618641,0.672408,14.486485,7.506505,1.573994,18.609307,-0.123029,0.468275,0.758920,IMERG,Per Hari (Daily),mm/hari
2,563,0.489756,0.513290,12.202751,7.198478,-1.888547,-22.328259,0.203143,0.327186,0.657231,PERSIANN,Per Hari (Daily),mm/hari
3,563,0.477916,0.517687,14.008585,9.219429,2.853512,33.737025,-0.050155,0.377277,0.683642,CHIRPS_FNL,Per Hari (Daily),mm/hari
4,563,0.442130,0.567662,12.847742,7.347441,-0.986568,-11.664177,0.116679,0.362066,0.633075,CHIRPS_RNL,Per Hari (Daily),mm/hari
5,563,0.385437,0.539447,16.534842,8.485212,0.186001,2.199088,-0.463069,0.362670,0.592224,ERA5,Per Hari (Daily),mm/hari
6,563,0.376672,0.533400,16.643812,8.567735,0.202233,2.391000,-0.482417,0.354334,0.584826,ERA5_LAND,Per Hari (Daily),mm/hari
7,563,0.350264,0.542938,15.120062,8.361700,-0.393385,-4.650985,-0.223410,0.345645,0.575334,GSMaP,Per Hari (Daily),mm/hari


,Produk,r_Hourly,r_Daily,rho_Hourly,rho_Daily,MAE_Hourly_mm_hr,MAE_Daily_mm_day,KGE_Hourly,KGE_Daily
0,IMERG,0.414100,0.618641,0.348455,0.672408,0.486667,7.506505,0.389503,0.468275
1,GSMaP,0.313250,0.350264,0.342305,0.542938,0.478878,8.361700,0.188612,0.345645
2,ERA5,0.104567,0.385437,0.212964,0.539447,0.579556,8.485212,0.080589,0.362670
3,ERA5_LAND,0.120297,0.376672,0.220470,0.533400,0.578657,8.567735,0.091573,0.354334
4,OYA,0.666375,0.759212,0.463397,0.724035,0.324752,4.886420,0.587658,0.676079


## 🖼️ 3. Visualisasi Hasil Analisis Multi-Skala

In [8]:
from IPython.display import Image, display

plots = [
    '01_bar_evaluasi_8satelit_vs_aws_harian.png',
    '02_scatter_hexbin_8satelit_vs_aws_harian.png',
    '03_perbandingan_scatter_jam_vs_hari.png',
    '04_bar_lonjakan_akurasi_jam_vs_hari.png',
    '05_siklus_diurnal_24jam_cuaca_hujan.png',
    '06_skor_kontingensi_deteksi_hujan_harian.png',
    '07_kurva_massa_ganda_harian.png',
    '08_heatmap_korelasi_harian_semua_produk.png',
    '09_heatmap_multimetrik_inter_model_harian.png',
    '10_heatmap_evaluasi_multimetrik_harian_vs_aws.png'
]

for p in plots:
    fp = os.path.join('Hasil_Analisis', p)
    if os.path.exists(fp):
        print(f"=== {p} ===")
        display(Image(fp))


=== 01_bar_evaluasi_8satelit_vs_aws_harian.png ===
<IPython.core.display.Image object>
=== 02_scatter_hexbin_8satelit_vs_aws_harian.png ===
<IPython.core.display.Image object>
=== 03_perbandingan_scatter_jam_vs_hari.png ===
<IPython.core.display.Image object>
=== 04_bar_lonjakan_akurasi_jam_vs_hari.png ===
<IPython.core.display.Image object>
=== 05_siklus_diurnal_24jam_cuaca_hujan.png ===
<IPython.core.display.Image object>
=== 06_skor_kontingensi_deteksi_hujan_harian.png ===
<IPython.core.display.Image object>
=== 07_kurva_massa_ganda_harian.png ===
<IPython.core.display.Image object>
=== 08_heatmap_korelasi_harian_semua_produk.png ===
<IPython.core.display.Image object>
=== 09_heatmap_multimetrik_inter_model_harian.png ===
<IPython.core.display.Image object>
=== 10_heatmap_evaluasi_multimetrik_harian_vs_aws.png ===
<IPython.core.display.Image object>


## 🎯 4. Kesimpulan & Rekomendasi
1. **NASA GPM IMERG** merupakan produk presipitasi harian terbaik terhadap observasi darat AWS IoT di Kebumen ($r = 0.619, ho = 0.642$).
2. Agregasi harian menghasilkan lonjakan korelasi sebesar $+49.5\%$ dibandingkan resolusi 1-jam.
3. Hujan konvektif sore hari (15:00–18:00 WIB) mendominasi kejadian hujan di Stasiun Jerukagung Kebumen.
